In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor, Pool

# CONFIG

In [2]:
DATA_PATH = "/kaggle/input/playground-series-s6e1/"
TARGET = "exam_score"
SEED = 42

# LOAD

In [3]:
train = pd.read_csv(DATA_PATH + "train.csv")
test  = pd.read_csv(DATA_PATH + "test.csv")
sub   = pd.read_csv(DATA_PATH + "sample_submission.csv")

y_raw = train[TARGET].values.astype(float)
X = train.drop(columns=[TARGET])
X_test = test.copy()

if "id" in X.columns:
    X = X.drop(columns=["id"])
    X_test = X_test.drop(columns=["id"])


# TARGET TRANSFORM (KEY)

In [4]:
qt = QuantileTransformer(
    n_quantiles=min(2000, len(y_raw)),
    output_distribution="normal",
    random_state=SEED
)
y = qt.fit_transform(y_raw.reshape(-1, 1)).ravel()

# CATEGORICAL


In [5]:
cat_cols = [c for c in X.columns if X[c].dtype == "object"]
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

for c in cat_cols:
    X[c] = X[c].astype(str).fillna("__MISSING__")
    X_test[c] = X_test[c].astype(str).fillna("__MISSING__")

# numeric fill
num_cols = [c for c in X.columns if c not in cat_cols]
X[num_cols] = X[num_cols].apply(pd.to_numeric, errors="coerce")
X_test[num_cols] = X_test[num_cols].apply(pd.to_numeric, errors="coerce")

X[num_cols] = X[num_cols].fillna(X[num_cols].median())
X_test[num_cols] = X_test[num_cols].fillna(X[num_cols].median())


# HOLDOUT SPLIT (VERY IMPORTANT)

In [6]:
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

train_pool = Pool(X_tr, y_tr, cat_features=cat_idx)
valid_pool = Pool(X_va, y_va, cat_features=cat_idx)


# MODEL (HEAVILY REGULARIZED)

In [7]:
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=12000,
    learning_rate=0.04,
    depth=7,
    l2_leaf_reg=10.0,
    min_data_in_leaf=120,
    random_strength=2.0,
    bagging_temperature=1.0,
    rsm=0.9,
    bootstrap_type="Bayesian",
    grow_policy="SymmetricTree",
    random_seed=SEED,
    od_type="Iter",
    od_wait=300,
    verbose=200,
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

0:	learn: 1.2905369	test: 1.2880140	best: 1.2880140 (0)	total: 496ms	remaining: 1h 39m 11s
200:	learn: 0.7491305	test: 0.7530514	best: 0.7530514 (200)	total: 1m 8s	remaining: 1h 6m 40s
400:	learn: 0.7451250	test: 0.7502570	best: 0.7502570 (400)	total: 2m 8s	remaining: 1h 1m 58s
600:	learn: 0.7415529	test: 0.7481910	best: 0.7481910 (600)	total: 3m 21s	remaining: 1h 3m 48s
800:	learn: 0.7389542	test: 0.7468814	best: 0.7468814 (800)	total: 4m 34s	remaining: 1h 3m 59s
1000:	learn: 0.7366972	test: 0.7459558	best: 0.7459558 (1000)	total: 5m 47s	remaining: 1h 3m 37s
1200:	learn: 0.7347323	test: 0.7453324	best: 0.7453324 (1200)	total: 7m	remaining: 1h 3m 5s
1400:	learn: 0.7329409	test: 0.7448604	best: 0.7448538 (1387)	total: 8m 13s	remaining: 1h 2m 10s
1600:	learn: 0.7313129	test: 0.7444141	best: 0.7444141 (1600)	total: 9m 26s	remaining: 1h 1m 19s
1800:	learn: 0.7296292	test: 0.7439970	best: 0.7439962 (1799)	total: 10m 40s	remaining: 1h 27s
2000:	learn: 0.7280333	test: 0.7437155	best: 0.743715

# VALID SCORE (TRANSFORM SPACE)

In [8]:
va_pred = model.predict(X_va)
print("Holdout RMSE (gauss space):",
      np.sqrt(mean_squared_error(y_va, va_pred)))


Holdout RMSE (gauss space): 0.7423182138427504


# PREDICT TEST

In [9]:
pred_test = model.predict(X_test)

# inverse target transform
pred_final = qt.inverse_transform(pred_test.reshape(-1, 1)).ravel()
pred_final = np.clip(pred_final, 0, 100)

sub[TARGET] = pred_final
sub.to_csv("submission.csv", index=False)

sub.head()


,id,exam_score
0,630000,71.600000
1,630001,70.700000
2,630002,92.124062
3,630003,55.500000
4,630004,48.698585
